In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict

In [18]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
FANTOM = DATA / "fantom_cat"
PREPROCESSING = Path("../preprocessing")
IPS = DATA_INITIAL / "deg/rnaseq_res_ips"
FIBRO = DATA_INITIAL / "deg/rnaseq_res_fibro"
IPS_INDEX = PREPROCESSING / "deg_index/ips_deg_index"
FIBRO_INDEX = PREPROCESSING / "deg_index/fibro_deg_index"
BIOMART = DATA / "biomart"
BIOMART_INITIAL = DATA_INITIAL / "biomart"

DATA_INITIAL = Path("../data_initial").resolve()
FANTOM_INITIAL = DATA_INITIAL / "fantom_cat"

In [4]:
to_ips = IPS / "EdgeR.table.all.target.tsv.gz"
ips_expr = pd.read_csv(to_ips, sep="\t")
ips_expr.head()

,DEG.geneID,DEG.HGNC,EdgeR.log2FC,EdgeR.log2CPM,EdgeR.pvalue,EdgeR.FDR,EdgeR.Zscore,target.gapmerID,target.geneID,target.HGNC
0,CATG00000000155,CATG00000000155.1,0.257141,3.591144,0.550588,1.0,0.620927,G0234787_01,ENSG00000234787,LINC00458
1,CATG00000001164,CATG00000001164.1,0.706131,3.805757,0.044323,1.0,1.757448,G0234787_01,ENSG00000234787,LINC00458
2,CATG00000001318,CATG00000001318.1,-0.074655,4.766566,0.913694,1.0,0.052701,G0234787_01,ENSG00000234787,LINC00458
3,CATG00000002817,CATG00000002817.1,-0.201767,3.841792,0.716487,1.0,-0.414103,G0234787_01,ENSG00000234787,LINC00458
4,CATG00000003248,CATG00000003248.1,-0.155609,4.592817,0.650460,1.0,0.026003,G0234787_01,ENSG00000234787,LINC00458


In [68]:
"ENSG00000273388" in ips_expr["DEG.geneID"]

False

In [69]:
"ENSG00000272975" in ips_expr["DEG.geneID"]

False

In [39]:
ips_deg_genes = set(ips_expr["DEG.geneID"].unique())
len(ips_deg_genes)

10868

In [6]:
to_fibro = FIBRO / "DESeq2_genes_ASO_all.tsv.gz"
fibro_expr = pd.read_csv(to_fibro, sep="\t")
fibro_expr.head(3)

,Description,ASO_CAGE,KD.geneSymbol,perturb_id,geneID,geneSymbol,baseMean,log2FC,pvalue,fdr,KD.tpm.ave,NC.tpm.ave,ZScore
0,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000200169,RNU5D-1,134.656674,-2.325717,1.640034e-11,2.548449e-07,3.026571,15.427402,-3.106205
1,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000138675,FGF5,1788.727654,-1.371879,7.814666e-11,6.071605e-07,75.054256,196.439355,-2.756217
2,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000252521,RNU5D-2P,108.388065,-1.925674,1.102746e-09,5.711854e-06,3.178508,12.249495,-2.534245


In [40]:
fibro_deg_genes = set(fibro_expr["geneID"].unique())
len(fibro_deg_genes)

21041

In [66]:
"ENSG00000273388" in fibro_expr["geneID"]

False

In [70]:
"ENSG00000272975" in fibro_expr["geneID"]

False

In [19]:
gtf_file_FANTOM_l2 = FANTOM_INITIAL / "FANTOM_CAT.lv2_permissive.gtf.gz" #
anno_FANTOM_l2 = pd.read_csv(gtf_file_FANTOM_l2, sep="\t", header=None)
anno_FANTOM_l2.head(2)

,0,1,2,3,4,5,6,7,8
0,chr1,FANTOM,gene,953490,954359,.,+,.,"gene_id ""CATG00000000002.1""; geneSuperClass ""a..."
1,chr1,FANTOM,transcript,953490,954359,.,+,.,"gene_id ""CATG00000000002.1""; transcript_id ""EN..."


In [22]:
anno_FANTOM_l2["ens_gene"] = anno_FANTOM_l2[8].apply(lambda x: x.split(" ")[1].replace('"', '').replace(';', '').split(".")[0])

In [23]:
anno_FANTOM_l2["unique_genome_reg_name"] = anno_FANTOM_l2["ens_gene"] + "-" + anno_FANTOM_l2[2]

In [41]:
fantom_l2_genes = set(anno_FANTOM_l2["ens_gene"].unique())
len(fantom_l2_genes)

124245

In [42]:
ips_delta = ips_deg_genes - fantom_l2_genes
len(ips_delta)

1

In [43]:
ips_delta

{'CATGR0000028081'}

In [45]:
fibro_delta = fibro_deg_genes - fantom_l2_genes
len(fibro_delta)

2

In [47]:
fibro_delta

{'CATGC0000023317', 'CATGE0000171943'}

In [48]:
to_overlay_GENCODEv25 = FANTOM_INITIAL / "FANTOM_CAT.overlay_GENCODEv25.gtf.gz" #
FANTOM_overlay_GENCODEv25 = pd.read_csv(to_overlay_GENCODEv25, sep="\t")
FANTOM_overlay_GENCODEv25.head(2)

,trnscptID,geneID,geneType,sameTrnscptStructure_withPartner_IDStr,end3Truncate_withPartner_IDStr,end3Truncate_withinQryOnly_IDStr,nonRedundant
0,ENST00000496771.5_1,ENSG00000000003.14_1,protein_coding,ENST00000496771.1,NaN,NaN,no
1,ENST00000373020.8_1,ENSG00000000003.14_1,protein_coding,ENST00000373020.4,NaN,NaN,no


In [49]:
FANTOM_overlay_GENCODEv25["sameTrnscptStructure_withPartner_IDStr"] = FANTOM_overlay_GENCODEv25["sameTrnscptStructure_withPartner_IDStr"].astype(str)
FANTOM_overlay_GENCODEv25["end3Truncate_withPartner_IDStr"] = FANTOM_overlay_GENCODEv25["end3Truncate_withPartner_IDStr"].astype(str)
FANTOM_overlay_GENCODEv25["end3Truncate_withinQryOnly_IDStr"] = FANTOM_overlay_GENCODEv25["end3Truncate_withinQryOnly_IDStr"].astype(str)

In [50]:
FANTOM_overlay_GENCODEv25.shape

(200052, 7)

In [51]:
FANTOM_overlay_GENCODEv25["geneID_without_v"] = FANTOM_overlay_GENCODEv25["geneID"].apply(lambda x: x.split(".")[0])

In [52]:
FANTOM_overlay_GENCODEv25["geneID_without_v"].nunique()

60162

In [56]:
mask = FANTOM_overlay_GENCODEv25["sameTrnscptStructure_withPartner_IDStr"].apply(lambda x: "CAT" in x)
FANTOM_overlay_GENCODEv25[mask]

,trnscptID,geneID,geneType,sameTrnscptStructure_withPartner_IDStr,end3Truncate_withPartner_IDStr,end3Truncate_withinQryOnly_IDStr,nonRedundant,geneID_without_v


In [57]:
FANTOM_overlay_GENCODEv25[["geneID_without_v","nonRedundant"]]

,geneID_without_v,nonRedundant
0,ENSG00000000003,no
1,ENSG00000000003,no
2,ENSG00000000003,no
3,ENSG00000000003,yes
4,ENSG00000000003,yes
...,...,...
200047,ENSG00000283695,yes
200048,ENSG00000283696,no
200049,ENSG00000283697,yes
200050,ENSG00000283698,yes
